# ADC: Microphone to Signal

In this part, we test the device's ability to record input and convert it into a signal. Each device is different and may have a different sampling rate. For example the Macbook Pro M1 has a sampling rate of 48000.

I personally use both the macbook pro and a Linux (ubuntu) machine. Therefore I have not tested this on Windows.

- If you use mac or windows, the sounddevice library in *requirements.txt* should be enough for audio capture.
- If you use linux, then you need to to install the extra dependency
```
apt install libportaudio2
```

To view all the information for microphone registration, please refer to the README.md in the audio folder

In [12]:
# avoid caching modules
%load_ext autoreload
%autoreload 2

import importlib
import numpy as np
import matplotlib.pyplot as plt

from audio.file_tools import save_audio, load_audio
from audio.recording import record_audio, list_input_devices 
from dsp.template_tools import rank_chords

# Import modules for audio processing
from pipeline.chord_guess import full_pipeline


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Before proceeding, it is best to know the device you are using and some information about it.

The `list_input_devices` lists all input devices that your device has registered. It is entirely possible for it to list multiple devices.

In [ ]:
list_input_devices()


[{'name': 'Black Pearl Microphone',
  'index': 1,
  'hostapi': 0,
  'max_input_channels': 1,
  'max_output_channels': 0,
  'default_low_input_latency': 0.12841666666666668,
  'default_low_output_latency': 0.01,
  'default_high_input_latency': 0.13775,
  'default_high_output_latency': 0.1,
  'default_samplerate': 48000.0},
 {'name': 'MacBook Pro Microphone',
  'index': 2,
  'hostapi': 0,
  'max_input_channels': 1,
  'max_output_channels': 0,
  'default_low_input_latency': 0.03285416666666666,
  'default_low_output_latency': 0.01,
  'default_high_input_latency': 0.0421875,
  'default_high_output_latency': 0.1,
  'default_samplerate': 48000.0},
 {'name': 'Microsoft Teams Audio',
  'index': 4,
  'hostapi': 0,
  'max_input_channels': 1,
  'max_output_channels': 1,
  'default_low_input_latency': 0.01,
  'default_low_output_latency': 0.0013333333333333333,
  'default_high_input_latency': 0.1,
  'default_high_output_latency': 0.010666666666666666,
  'default_samplerate': 48000.0}]

Now let's capture a sound, depending on your device try to capture something. This function alone will take the default listening device. For this test, we will testing how well the audio has captured a D chord.

- Chosen chord: D major chord
- Duration: 2 seconds

As a reminder the `full_pipeline` attempts to guess the chord.
1. Mono: Convert the sample recorded sound into mono
2. Start Detection: Detect the start of the recorded sound to avoid processing silence
3. Hann Window: Apply windowing (Hann Window) to concentrate the middle of the sample.
4. FFT: Perform Fast Fourier Transform to create a spectrum of detected frequencies.
5. Peak Detection: Attempt to find notable peaks along the frequency spectrum.
6. Chroma: Create a bin of 12 to label frequencies into notes, this is also where MIDI conversion happens.
7. Template Matching: Using cosine similarity and a bank of templates, find the best matches for chords

In [23]:
# Recording phase

# Record for 2 seconds of audio, but have a countdown
import time
for i in range(3, 0, -1):
    print(i)
    time.sleep(1)

print("Recording...")

samples, rate = record_audio(2)

# save recording for debugging
save_audio("./recordings/recording.wav", samples, rate)

print("Done.")

# Process the recorded audio
# Create an empty dictionary for outputs
outputs = {}

outputs = full_pipeline(samples, rate)

# Print everything from the outputs
print(outputs['chord'])

# Get the ranked chroma
ranked = rank_chords(outputs['chroma'])[:5]

print("\nTop 5:")
print(ranked)

3
2
1
Recording...
Done.
D

Top 5:
[('D', 0.9239808803801804), ('D7', 0.8635495779664156), ('Dmaj7', 0.8496366389811466), ('Bm7', 0.8001909150203469), ('F#dim', 0.7957538861710394)]


From the results seen above, the D chord was correctly guessed. The second closest guess was D7.

Next, let's try just loading it directly. For this code block to work, you probably need to adjust the path. Note that only .wav files are accepted. As a reminder, this stack only saves .wav files as well.

For this example, I downloaded an F chord guitar strum, the source is on the following link:
https://pixabay.com/sound-effects/search/f-chord/



In [8]:
# Use the load functions
loaded_samples, loaded_rate = load_audio("recordings/downloaded/f-chord.wav")

# Additionally print some information
print(loaded_samples.shape, loaded_rate, loaded_samples.dtype, np.abs(loaded_samples).max())


(177408, 2) 44100 float32 0.49902344


Now, let's run the complete pipeline against this loaded recording. When loaded, all that is returned are the samples and the sample rate.

In [18]:
# Process the loaded audio (now samples)
# Create an empty dictionary for the loaded audio outputs
outputs_from_load = {}

outputs_from_load = full_pipeline(loaded_samples, loaded_rate)

# Print everything from the outputs
print(outputs_from_load['chord'])

# Attempt to get the ranked chroma, only the top 5
ranked = rank_chords(outputs_from_load['chroma'])[:5]

print("\nTop 5:")
print(ranked)


F5

Top 5:
[('F5', 0.9155351658155833), ('F', 0.866965918223521), ('Csus4', 0.8055279631060082), ('Dm7', 0.7508145093968713), ('Fmaj7', 0.7508145093968713)]


The guessed chord was an F5. However, the second best match was indeed the F chord. 